# Semantic Model Recommender (Lakehouse → Direct Lake)

This Fabric notebook inspects the Delta tables in a lakehouse, profiles them, and asks an LLM
to recommend a **Power BI semantic model**: fact vs dimension tables, relationships, base DAX
measures, columns to hide, display names, and a date dimension. It then generates the model as
**TMDL (Direct Lake)** and can optionally deploy it to the workspace through the Fabric REST API.

**Pipeline**

1. Configure (cell below) — lakehouse, LLM provider, model name.
2. Profile tables — row counts, cardinality, null rates, candidate keys, sample values.
3. Detect relationship candidates — name matching + value-overlap checks between tables.
4. Ask the LLM for a structured recommendation (JSON).
5. Validate the recommendation against the real schema (no invented tables/columns).
6. Generate TMDL and write it to `Files/semantic_model_recommendations/<model>/`.
7. (Optional) Deploy the semantic model to this workspace.

**LLM providers supported**

| `LLM_PROVIDER`   | What it uses                                                            | Needs a key? |
|------------------|-------------------------------------------------------------------------|--------------|
| `fabric_openai`  | Azure OpenAI built into Fabric (capacity must have Copilot / AI enabled) | No           |
| `azure_openai`   | Your own Azure OpenAI resource                                           | Yes          |
| `anthropic`      | Claude via the Anthropic API                                             | Yes          |

Keys can be supplied inline (dev only) or read from Azure Key Vault via `notebookutils.credentials.getSecret`.

> The LLM only ever sees **metadata and a handful of sample values** per column, never full tables.
> Set `SEND_SAMPLE_VALUES = False` to send schema and statistics only.

In [ ]:
# ── 1. Configuration ────────────────────────────────────────────────────────
LAKEHOUSE_NAME   = None      # None = the notebook's default (attached) lakehouse
SCHEMA_NAME      = None      # e.g. "dbo" for schema-enabled lakehouses; None otherwise
TABLE_INCLUDE    = []        # [] = all Delta tables; otherwise a list of table names
TABLE_EXCLUDE    = []        # tables to skip (staging, bronze, etc.)
MAX_TABLES       = 40
SAMPLE_ROWS      = 200_000   # per-table sample used for cardinality / overlap statistics
SEND_SAMPLE_VALUES = True    # include up to 5 example values per column in the LLM prompt

# LLM
LLM_PROVIDER     = "fabric_openai"   # "fabric_openai" | "azure_openai" | "anthropic"
LLM_MODEL        = "gpt-4o"          # Azure: deployment name.  Anthropic: e.g. "claude-sonnet-5"
AZURE_OPENAI_ENDPOINT = ""           # only for "azure_openai", e.g. https://my-aoai.openai.azure.com/
AZURE_OPENAI_API_VERSION = "2024-10-21"
AZURE_OPENAI_API_KEY  = ""           # leave "" to read from Key Vault
ANTHROPIC_API_KEY     = ""           # leave "" to read from Key Vault
KEYVAULT_URL          = ""           # e.g. https://my-kv.vault.azure.net/
KEYVAULT_SECRET_NAME  = ""           # secret holding the API key for the chosen provider

# Business context helps the LLM name things and pick measures. Optional but recommended.
BUSINESS_CONTEXT = """
(Describe the domain, e.g. "Retail sales for a sporting goods chain. Key KPIs: net sales,
units, gross margin, store count. Fiscal year starts in February. Reports are by store,
product category and week.")
"""

# Output
MODEL_NAME            = None         # None = "<lakehouse> Recommended Model"
DIRECT_LAKE_MODE      = "onelake"    # "onelake" (recommended) | "sql_endpoint"
CREATE_DATE_TABLE     = True         # create a dim_date Delta table if the LLM recommends one and none exists
DATE_TABLE_NAME       = "dim_date"
WRITE_TMDL            = True
DEPLOY_TO_WORKSPACE   = False        # True = create the semantic model in this workspace via REST API

## 2. Resolve workspace / lakehouse context

In [ ]:
import json, re, uuid, base64, time, textwrap, math
from pyspark.sql import functions as F, types as T

ctx = notebookutils.runtime.context
WORKSPACE_ID = ctx["currentWorkspaceId"]

def _get(obj, key):
    return obj[key] if isinstance(obj, dict) else getattr(obj, key)

if LAKEHOUSE_NAME:
    _lh = notebookutils.lakehouse.get(LAKEHOUSE_NAME, WORKSPACE_ID)
    LAKEHOUSE_ID = _get(_lh, "id")
else:
    LAKEHOUSE_ID   = ctx.get("defaultLakehouseId")
    LAKEHOUSE_NAME = ctx.get("defaultLakehouseName")
    if not LAKEHOUSE_ID:
        raise RuntimeError("No default lakehouse attached. Attach one or set LAKEHOUSE_NAME.")

MODEL_NAME = MODEL_NAME or f"{LAKEHOUSE_NAME} Recommended Model"
DB = f"`{LAKEHOUSE_NAME}`" + (f".`{SCHEMA_NAME}`" if SCHEMA_NAME else "")

print(f"Workspace : {WORKSPACE_ID}")
print(f"Lakehouse : {LAKEHOUSE_NAME} ({LAKEHOUSE_ID})")
print(f"Spark DB  : {DB}")
print(f"Model     : {MODEL_NAME}  [Direct Lake / {DIRECT_LAKE_MODE}]")

## 3. Discover and profile the Delta tables

In [ ]:
COMPLEX_PREFIXES = ("struct", "array", "map")
KEY_HINTS = ("id", "key", "code", "sk", "nbr", "num", "no")

def norm(s):
    return re.sub(r"[^a-z0-9]", "", s.lower())

def looks_like_key(col):
    n = col.lower()
    return any(n.endswith(h) or n.endswith("_" + h) for h in KEY_HINTS)

def list_delta_tables():
    tables = [r["tableName"] for r in spark.sql(f"SHOW TABLES IN {DB}").collect()
              if not r["isTemporary"]]
    try:
        views = {r["viewName"] for r in spark.sql(f"SHOW VIEWS IN {DB}").collect()}
    except Exception:
        views = set()
    out = []
    for t in tables:
        if t in views:
            print(f"  skip view  {t}  (Direct Lake needs Delta tables, views fall back to DirectQuery)")
            continue
        if TABLE_INCLUDE and t not in TABLE_INCLUDE:
            continue
        if t in TABLE_EXCLUDE:
            continue
        out.append(t)
    return out[:MAX_TABLES]

def spark_to_tmdl_type(t):
    t = t.lower()
    if t in ("tinyint", "smallint", "int", "integer", "bigint", "long", "short", "byte"): return "int64"
    if t in ("double", "float"): return "double"
    if t.startswith("decimal"): return "decimal"
    if t in ("string", "varchar", "char"): return "string"
    if t == "boolean": return "boolean"
    if t in ("date", "timestamp", "timestamp_ntz"): return "dateTime"
    if t == "binary": return "binary"
    return None  # unsupported in Direct Lake

def profile_table(name):
    df = spark.table(f"{DB}.`{name}`")
    total = df.count()
    frac = 1.0 if total <= SAMPLE_ROWS else SAMPLE_ROWS / total
    sample = df if frac >= 1.0 else df.sample(fraction=frac, seed=42)
    sample = sample.cache()
    n = sample.count() or 1

    cols, aggs = [], []
    for f in df.schema.fields:
        c, st = f.name, f.dataType.simpleString()
        tmdl_t = spark_to_tmdl_type(st)
        info = {"name": c, "spark_type": st, "tmdl_type": tmdl_t, "supported": tmdl_t is not None}
        cols.append(info)
        if not info["supported"]:
            continue
        col = F.col(f"`{c}`")
        aggs += [F.approx_count_distinct(col).alias(f"{c}__nd"),
                 F.sum(col.isNull().cast("int")).alias(f"{c}__nulls")]
        if tmdl_t in ("int64", "double", "decimal", "dateTime"):
            aggs += [F.min(col).alias(f"{c}__min"), F.max(col).alias(f"{c}__max")]
    stats = sample.agg(*aggs).collect()[0].asDict() if aggs else {}

    sample_rows = sample.limit(5).collect()
    for info in cols:
        c = info["name"]
        if not info["supported"]:
            continue
        nd, nulls = int(stats.get(f"{c}__nd", 0)), int(stats.get(f"{c}__nulls", 0))
        info.update({
            "distinct_ratio": round(nd / n, 4),
            "null_ratio": round(nulls / n, 4),
            "approx_distinct": nd,
            "min": str(stats.get(f"{c}__min")) if f"{c}__min" in stats else None,
            "max": str(stats.get(f"{c}__max")) if f"{c}__max" in stats else None,
            "is_candidate_key": nulls == 0 and nd / n >= 0.98 and n > 1,
            "looks_like_key": looks_like_key(c),
            "is_numeric": info["tmdl_type"] in ("int64", "double", "decimal"),
            "is_date": info["tmdl_type"] == "dateTime",
        })
        if SEND_SAMPLE_VALUES:
            info["samples"] = [str(r[c])[:40] for r in sample_rows if r[c] is not None][:5]

    prof = {
        "table": name,
        "row_count": total,
        "sampled_rows": n,
        "columns": cols,
        "candidate_keys": [c["name"] for c in cols if c.get("is_candidate_key")],
        "unsupported_columns": [c["name"] for c in cols if not c["supported"]],
    }
    return prof, sample

TABLES = list_delta_tables()
print(f"Profiling {len(TABLES)} tables: {TABLES}")
PROFILES, SAMPLES = {}, {}
for t in TABLES:
    t0 = time.time()
    PROFILES[t], SAMPLES[t] = profile_table(t)
    p = PROFILES[t]
    print(f"  {t:<40} rows={p['row_count']:>12,}  cols={len(p['columns']):>3}  "
          f"keys={p['candidate_keys']}  ({time.time()-t0:.1f}s)")

## 4. Detect relationship candidates (name match + value overlap)

In [ ]:
def overlap_ratio(from_t, from_c, to_t, to_c, limit=50_000):
    """Share of distinct sampled FROM values that exist in TO (both cast to string)."""
    a = SAMPLES[from_t].select(F.col(f"`{from_c}`").cast("string").alias("v")).dropna().distinct().limit(limit)
    b = SAMPLES[to_t].select(F.col(f"`{to_c}`").cast("string").alias("v")).dropna().distinct()
    a_n = a.count()
    if a_n == 0:
        return 0.0
    return round(a.join(b, "v", "left_semi").count() / a_n, 4)

def relationship_candidates():
    keys = [(t, k) for t, p in PROFILES.items() for k in p["candidate_keys"]]
    cands = []
    for ft, fp in PROFILES.items():
        for col in fp["columns"]:
            if not col["supported"]:
                continue
            fc, nfc = col["name"], norm(col["name"])
            for (tt, tk) in keys:
                if tt == ft:
                    continue
                ntk, ntt = norm(tk), norm(tt)
                name_match = (nfc == ntk) or (nfc == ntt + ntk) or (nfc == ntt + "id") or \
                             (nfc == ntt + "key") or (looks_like_key(fc) and ntt in nfc)
                if not name_match:
                    continue
                if fp["row_count"] < PROFILES[tt]["row_count"] and col.get("is_candidate_key"):
                    continue  # avoid PK-to-PK pairs in the wrong direction
                ov = overlap_ratio(ft, fc, tt, tk)
                if ov >= 0.5:
                    cands.append({"from_table": ft, "from_column": fc, "to_table": tt, "to_column": tk,
                                  "value_overlap": ov, "name_match": True})
    return sorted(cands, key=lambda c: -c["value_overlap"])

REL_CANDIDATES = relationship_candidates()
print(f"{len(REL_CANDIDATES)} relationship candidates")
for c in REL_CANDIDATES:
    print(f"  {c['from_table']}.{c['from_column']}  →  {c['to_table']}.{c['to_column']}   overlap={c['value_overlap']:.0%}")

## 5. Heuristic role hints (the LLM makes the final call)

In [ ]:
def role_hint(p):
    supported = [c for c in p["columns"] if c["supported"]]
    numeric_non_key = [c for c in supported if c["is_numeric"] and not c["looks_like_key"] and not c["is_candidate_key"]]
    fk_cols = {c["from_column"] for c in REL_CANDIDATES if c["from_table"] == p["table"]}
    dates = [c for c in supported if c["is_date"]]
    n = p["table"].lower()
    if n.startswith(("fact", "f_", "fct")) or n.endswith("_fact"):
        return "fact"
    if n.startswith(("dim", "d_")) or n.endswith("_dim"):
        return "dimension"
    if len(fk_cols) >= 2 and (numeric_non_key or dates):
        return "fact"
    if p["candidate_keys"] and len(numeric_non_key) <= 2:
        return "dimension"
    return "unknown"

for t, p in PROFILES.items():
    p["role_hint"] = role_hint(p)
    print(f"  {t:<40} {p['role_hint']}")

## 6. LLM client

In [ ]:
def _secret():
    if KEYVAULT_URL and KEYVAULT_SECRET_NAME:
        return notebookutils.credentials.getSecret(KEYVAULT_URL, KEYVAULT_SECRET_NAME)
    return None

def _openai_chat(client, system_prompt, user_prompt, max_tokens):
    msgs = [{"role": "system", "content": system_prompt}, {"role": "user", "content": user_prompt}]
    try:
        r = client.chat.completions.create(model=LLM_MODEL, temperature=0, max_tokens=max_tokens,
                                           response_format={"type": "json_object"}, messages=msgs)
    except Exception as e:  # older deployments / API versions without JSON mode
        print(f"JSON mode unavailable ({type(e).__name__}); retrying without response_format")
        r = client.chat.completions.create(model=LLM_MODEL, temperature=0, max_tokens=max_tokens, messages=msgs)
    return r.choices[0].message.content

def call_llm(system_prompt, user_prompt, max_tokens=8000):
    if LLM_PROVIDER == "fabric_openai":
        # Azure OpenAI pre-built into Fabric: token + workload endpoint, no key needed.
        from openai import AzureOpenAI
        from synapse.ml.mlflow import get_mlflow_env_config
        cfg = get_mlflow_env_config()
        client = AzureOpenAI(
            azure_endpoint=cfg.workload_endpoint + "cognitive/openai/",
            api_key=cfg.driver_aad_token,
            api_version=AZURE_OPENAI_API_VERSION,
        )
        return _openai_chat(client, system_prompt, user_prompt, max_tokens)

    if LLM_PROVIDER == "azure_openai":
        from openai import AzureOpenAI
        client = AzureOpenAI(azure_endpoint=AZURE_OPENAI_ENDPOINT,
                             api_key=AZURE_OPENAI_API_KEY or _secret(),
                             api_version=AZURE_OPENAI_API_VERSION)
        return _openai_chat(client, system_prompt, user_prompt, max_tokens)

    if LLM_PROVIDER == "anthropic":
        try:
            import anthropic
        except ImportError:
            get_ipython().run_line_magic("pip", "install -q anthropic")
            import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY or _secret())
        r = client.messages.create(
            model=LLM_MODEL, max_tokens=max_tokens, temperature=0,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}])
        return "".join(b.text for b in r.content if getattr(b, "type", "") == "text")

    raise ValueError(f"Unknown LLM_PROVIDER {LLM_PROVIDER!r}")

def parse_json(text):
    text = text.strip()
    m = re.search(r"```(?:json)?\s*(\{.*\})\s*```", text, re.S)
    if m:
        text = m.group(1)
    start, end = text.find("{"), text.rfind("}")
    return json.loads(text[start:end + 1])

## 7. Ask the LLM for a semantic model recommendation

In [ ]:
SYSTEM_PROMPT = """You are a senior Power BI / Microsoft Fabric data modeler. You design star-schema
semantic models in Direct Lake mode over lakehouse Delta tables.

Hard rules:
- Use ONLY tables and columns that appear in the profile. Never invent names.
- Direct Lake constraints: no calculated columns, no calculated tables, no DAX-based date tables,
  no Power Query transformations. Anything derived must already exist as a Delta table/column.
  If a date dimension is needed and none exists, set date_dimension.create = true and describe it.
- Relationships: single-column keys, many-to-one from fact to dimension, single cross-filter
  direction unless there is a strong reason. Both columns must have the same data type.
- Hide foreign-key columns on facts and surrogate keys on dimensions. Set summarizeBy none on keys.
- Measures: explicit DAX measures for every additive numeric fact column (SUM), plus row counts and
  the 3-6 most useful business KPIs implied by the data (ratios, averages). Provide formatString.
- Display names: business-friendly (title case, no prefixes like dim_/fact_, no underscores).
- Exclude tables that are clearly staging/bronze/technical or that have no analytical value, and say why.
Return ONLY a JSON object (no prose) matching the schema requested."""

OUTPUT_SCHEMA = """{
  "model_summary": "2-3 sentence description of the recommended model and its grain",
  "tables": [
    {"name": "<exact lakehouse table name>", "role": "fact|dimension",
     "display_name": "...", "description": "...",
     "primary_key": "<column or null>",
     "hidden_columns": ["..."],
     "column_display_names": {"<column>": "<display name>"},
     "column_format_strings": {"<column>": "<format string>"},
     "include": true}
  ],
  "excluded_tables": [{"name": "...", "reason": "..."}],
  "relationships": [
    {"from_table": "...", "from_column": "...", "to_table": "...", "to_column": "...",
     "cardinality": "manyToOne|oneToOne", "cross_filter": "single|both", "is_active": true,
     "rationale": "..."}
  ],
  "measures": [
    {"table": "<home table>", "name": "...", "dax": "...", "format_string": "...", "description": "..."}
  ],
  "date_dimension": {"exists": true, "table": "<name or null>", "date_column": "<col or null>",
                     "create": false, "mark_as_date_table": true,
                     "fact_date_columns": [{"table": "...", "column": "..."}]},
  "warnings": ["Direct Lake / data-quality concerns the modeler should know"],
  "open_questions": ["things to confirm with the business"]
}"""

def build_user_prompt():
    slim = []
    for t, p in PROFILES.items():
        slim.append({
            "table": t, "row_count": p["row_count"], "role_hint": p["role_hint"],
            "candidate_keys": p["candidate_keys"],
            "unsupported_columns": p["unsupported_columns"],
            "columns": [
                {k: v for k, v in c.items()
                 if k in ("name", "spark_type", "distinct_ratio", "null_ratio", "approx_distinct",
                          "min", "max", "is_candidate_key", "looks_like_key", "samples")}
                for c in p["columns"] if c["supported"]
            ],
        })
    return f"""BUSINESS CONTEXT
{BUSINESS_CONTEXT.strip()}

LAKEHOUSE: {LAKEHOUSE_NAME}
TABLE PROFILES (JSON):
{json.dumps(slim, indent=1, default=str)}

RELATIONSHIP CANDIDATES FROM VALUE OVERLAP (fraction of fact-side values found in dimension key):
{json.dumps(REL_CANDIDATES, indent=1)}

Recommend the semantic model. Respond with ONLY a JSON object using exactly this schema:
{OUTPUT_SCHEMA}"""

user_prompt = build_user_prompt()
print(f"Prompt size: ~{len(user_prompt)//4:,} tokens")
raw = call_llm(SYSTEM_PROMPT, user_prompt)
REC = parse_json(raw)
print(json.dumps(REC, indent=2)[:4000])

## 8. Validate the recommendation against the real schema

In [ ]:
def validate(rec):
    issues = []
    tables_by_name = {t: p for t, p in PROFILES.items()}
    col_types = {t: {c["name"]: c["tmdl_type"] for c in p["columns"] if c["supported"]} for t, p in PROFILES.items()}

    kept_tables = []
    for t in rec.get("tables", []):
        if t["name"] not in tables_by_name:
            issues.append(f"DROPPED table '{t['name']}': not in lakehouse"); continue
        if not t.get("include", True):
            continue
        cols = col_types[t["name"]]
        if t.get("primary_key") and t["primary_key"] not in cols:
            issues.append(f"{t['name']}: primary_key '{t['primary_key']}' not found, cleared"); t["primary_key"] = None
        t["hidden_columns"] = [c for c in t.get("hidden_columns", []) if c in cols]
        t["column_display_names"] = {k: v for k, v in t.get("column_display_names", {}).items() if k in cols}
        t["column_format_strings"] = {k: v for k, v in t.get("column_format_strings", {}).items() if k in cols}
        kept_tables.append(t)
    rec["tables"] = kept_tables
    kept_names = {t["name"] for t in kept_tables}

    rels = []
    for r in rec.get("relationships", []):
        ft, fc, tt, tc = r.get("from_table"), r.get("from_column"), r.get("to_table"), r.get("to_column")
        if ft not in kept_names or tt not in kept_names:
            issues.append(f"DROPPED relationship {ft}.{fc}→{tt}.{tc}: table not in model"); continue
        if fc not in col_types[ft] or tc not in col_types[tt]:
            issues.append(f"DROPPED relationship {ft}.{fc}→{tt}.{tc}: column not found"); continue
        if col_types[ft][fc] != col_types[tt][tc]:
            issues.append(f"WARNING relationship {ft}.{fc}→{tt}.{tc}: type mismatch "
                          f"{col_types[ft][fc]} vs {col_types[tt][tc]} (fix in lakehouse)")
        rels.append(r)
    rec["relationships"] = rels

    ident = re.compile(r"'?([A-Za-z0-9_ ]+?)'?\[([^\]]+)\]")
    measures = []
    for m in rec.get("measures", []):
        if m.get("table") not in kept_names:
            issues.append(f"DROPPED measure '{m.get('name')}': home table missing"); continue
        bad = [f"{t}[{c}]" for t, c in ident.findall(m.get("dax", ""))
               if t in col_types and c not in col_types[t] and c not in {x["name"] for x in rec["measures"]}]
        if bad:
            issues.append(f"WARNING measure '{m['name']}' references unknown columns {bad}")
        measures.append(m)
    rec["measures"] = measures
    return rec, issues

REC, VALIDATION_ISSUES = validate(REC)
print("\n".join(VALIDATION_ISSUES) or "No validation issues.")

## 9. (Optional) Create a date dimension Delta table if recommended

In [ ]:
dd = REC.get("date_dimension", {}) or {}
if CREATE_DATE_TABLE and dd.get("create") and DATE_TABLE_NAME not in PROFILES:
    all_min = min(c["min"] for t, p in PROFILES.items() for c in p["columns"] if c.get("is_date") and c.get("min"))
    all_max = max(c["max"] for t, p in PROFILES.items() for c in p["columns"] if c.get("is_date") and c.get("max"))
    start = f"{all_min[:4]}-01-01"; end = f"{all_max[:4]}-12-31"
    print(f"Creating {DATE_TABLE_NAME} from {start} to {end}")
    spark.sql(f"""
        CREATE OR REPLACE TABLE {DB}.`{DATE_TABLE_NAME}` USING DELTA AS
        SELECT d AS Date,
               CAST(date_format(d, 'yyyyMMdd') AS INT) AS DateKey,
               year(d) AS Year, quarter(d) AS Quarter, month(d) AS MonthNumber,
               date_format(d, 'MMMM') AS MonthName, date_format(d, 'MMM') AS MonthShort,
               CONCAT(year(d), '-Q', quarter(d)) AS YearQuarter,
               CAST(date_format(d, 'yyyyMM') AS INT) AS YearMonthNumber,
               date_format(d, 'yyyy MMM') AS YearMonth,
               weekofyear(d) AS WeekOfYear, dayofweek(d) AS DayOfWeekNumber,
               date_format(d, 'EEEE') AS DayName, day(d) AS DayOfMonth,
               CASE WHEN dayofweek(d) IN (1, 7) THEN true ELSE false END AS IsWeekend
        FROM (SELECT explode(sequence(to_date('{start}'), to_date('{end}'), interval 1 day)) AS d)
    """)
    PROFILES[DATE_TABLE_NAME], SAMPLES[DATE_TABLE_NAME] = profile_table(DATE_TABLE_NAME)
    PROFILES[DATE_TABLE_NAME]["role_hint"] = "dimension"
    REC["tables"].append({"name": DATE_TABLE_NAME, "role": "dimension", "display_name": "Date",
                          "description": "Calendar date dimension", "primary_key": "Date",
                          "hidden_columns": ["DateKey", "MonthNumber", "YearMonthNumber", "DayOfWeekNumber"],
                          "column_display_names": {}, "column_format_strings": {}, "include": True})
    dd.update({"exists": True, "table": DATE_TABLE_NAME, "date_column": "Date", "create": False})
    # Link fact date columns to the new table (match on dateTime type; DateKey ints also supported)
    col_types = {t: {c["name"]: c["tmdl_type"] for c in p["columns"] if c["supported"]} for t, p in PROFILES.items()}
    has_active = set()
    for fd in dd.get("fact_date_columns", []):
        t, c = fd.get("table"), fd.get("column")
        if t in col_types and c in col_types[t]:
            to_col = "Date" if col_types[t][c] == "dateTime" else "DateKey"
            REC["relationships"].append({"from_table": t, "from_column": c, "to_table": DATE_TABLE_NAME,
                                         "to_column": to_col, "cardinality": "manyToOne",
                                         "cross_filter": "single", "is_active": t not in has_active,
                                         "rationale": "Date dimension"})
            has_active.add(t)
            if col_types[t][c] == "dateTime" and PROFILES[t]["columns"] and \
               any(x["name"] == c and x["spark_type"].startswith("timestamp") for x in PROFILES[t]["columns"]):
                REC.setdefault("warnings", []).append(
                    f"{t}.{c} is a timestamp; the relationship to {DATE_TABLE_NAME}[Date] only matches rows "
                    f"at midnight. Add a DATE-typed column (or DateKey int) in the lakehouse and rerun.")
    REC["date_dimension"] = dd
    print(f"Added {DATE_TABLE_NAME} to the model with {len(dd.get('fact_date_columns', []))} date relationship(s).")
else:
    print("No date table created.", "(exists)" if dd.get("exists") else "")

## 10. Recommendation summary

In [ ]:
def summary_markdown(rec):
    out = [f"# {MODEL_NAME}", "", rec.get("model_summary", ""), "", "## Tables", "",
           "| Table | Role | Display name | Primary key | Hidden columns |", "|---|---|---|---|---|"]
    for t in rec["tables"]:
        out.append(f"| {t['name']} | {t['role']} | {t.get('display_name','')} | {t.get('primary_key') or ''} | "
                   f"{', '.join(t.get('hidden_columns', []))} |")
    if rec.get("excluded_tables"):
        out += ["", "## Excluded tables", ""] + [f"- **{e['name']}** — {e['reason']}" for e in rec["excluded_tables"]]
    out += ["", "## Relationships", "", "| From | To | Cardinality | Cross filter | Active |", "|---|---|---|---|---|"]
    for r in rec["relationships"]:
        out.append(f"| {r['from_table']}[{r['from_column']}] | {r['to_table']}[{r['to_column']}] | "
                   f"{r.get('cardinality','manyToOne')} | {r.get('cross_filter','single')} | {r.get('is_active', True)} |")
    out += ["", "## Measures", "", "| Table | Measure | DAX | Format |", "|---|---|---|---|"]
    for m in rec["measures"]:
        dax = m["dax"].replace("|", "\\|").replace("\n", " ")
        out.append(f"| {m['table']} | {m['name']} | `{dax}` | {m.get('format_string','')} |")
    if rec.get("warnings"):
        out += ["", "## Warnings", ""] + [f"- {w}" for w in rec["warnings"]]
    if VALIDATION_ISSUES:
        out += ["", "## Validation issues", ""] + [f"- {w}" for w in VALIDATION_ISSUES]
    if rec.get("open_questions"):
        out += ["", "## Open questions for the business", ""] + [f"- {q}" for q in rec["open_questions"]]
    return "\n".join(out)

SUMMARY_MD = summary_markdown(REC)
try:
    from IPython.display import Markdown, display
    display(Markdown(SUMMARY_MD))
except Exception:
    print(SUMMARY_MD)

## 11. Generate TMDL (Direct Lake)

Produces the parts of a Fabric semantic model definition:
`definition.pbism`, `definition/database.tmdl`, `definition/model.tmdl`, `definition/expressions.tmdl`,
`definition/relationships.tmdl`, `definition/tables/<table>.tmdl`.

In [ ]:
def q(name):
    return name if re.fullmatch(r"[A-Za-z0-9_]+", name) else "'" + name.replace("'", "''") + "'"

def lt():
    return str(uuid.uuid4())

def get_sql_endpoint():
    token = notebookutils.credentials.getToken("pbi")
    import requests
    r = requests.get(f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/lakehouses/{LAKEHOUSE_ID}",
                     headers={"Authorization": f"Bearer {token}"})
    r.raise_for_status()
    props = r.json()["properties"]["sqlEndpointProperties"]
    return props["connectionString"], props["id"]

def display_maps(rec):
    """Lakehouse name -> model (display) name for tables and columns."""
    tdisp = {t["name"]: (t.get("display_name") or t["name"]).strip() for t in rec["tables"]}
    cdisp = {t["name"]: {c["name"]: (t.get("column_display_names") or {}).get(c["name"]) or c["name"]
                         for c in PROFILES[t["name"]]["columns"]} for t in rec["tables"]}
    # de-duplicate display names inside a table
    for t, m in cdisp.items():
        seen = set()
        for src, d in m.items():
            if d in seen:
                m[src] = src
            seen.add(m[src])
    return tdisp, cdisp

def rewrite_dax(dax, tdisp, cdisp):
    """Rewrite Table[Column] references from lakehouse names to model display names."""
    for src_t, disp_t in tdisp.items():
        for src_c, disp_c in cdisp.get(src_t, {}).items():
            dax = re.sub(rf"(?:'{re.escape(src_t)}'|(?<![\w'\]]){re.escape(src_t)})\[{re.escape(src_c)}\]",
                         f"{q(disp_t)}[{disp_c}]", dax)
        if disp_t != src_t:
            # Table[Col] with an already-rewritten column, and bare table refs such as COUNTROWS(fact_sales)
            # or ALL(fact_sales). Skip anything followed by "(" so function names are never touched.
            dax = re.sub(rf"(?:'{re.escape(src_t)}'|(?<![\w'\]]){re.escape(src_t)}(?![\w']))(?!\s*\()",
                         q(disp_t), dax)
    return dax

def build_tmdl(rec):
    parts = {}
    tables = rec["tables"]
    table_names = [t["name"] for t in tables]
    tdisp, cdisp = display_maps(rec)
    measures_by_table = {}
    for m in rec["measures"]:
        m["dax_deployed"] = rewrite_dax(m["dax"].strip(), tdisp, cdisp)
        measures_by_table.setdefault(m["table"], []).append(m)
    dd = rec.get("date_dimension") or {}

    # -- expressions.tmdl (the Direct Lake source)
    if DIRECT_LAKE_MODE == "onelake":
        expr_name = "DirectLakeOneLake"
        expr_body = (f'AzureStorage.DataLake("https://onelake.dfs.fabric.microsoft.com/'
                     f'{WORKSPACE_ID}/{LAKEHOUSE_ID}", [HierarchicalNavigation=true])')
        compat = 1702
    else:
        expr_name = "DatabaseQuery"
        server, dbid = get_sql_endpoint()
        expr_body = f'let\n\t\t\tdatabase = Sql.Database("{server}", "{dbid}")\n\t\tin\n\t\t\tdatabase'
        compat = 1604
    parts["definition/expressions.tmdl"] = (
        f"expression {expr_name} =\n\t\t{expr_body}\n"
        f"\tlineageTag: {lt()}\n\n"
        f"\tannotation PBI_IncludeFutureArtifacts = False\n")

    # -- database.tmdl / model.tmdl
    parts["definition/database.tmdl"] = f"database\n\tcompatibilityLevel: {compat}\n"
    model = ["model Model",
             "\tculture: en-US",
             "\tdefaultPowerBIDataSourceVersion: powerBI_V3",
             "\tsourceQueryCulture: en-US",
             "\tdataAccessOptions",
             "\t\tlegacyRedirects",
             "\t\treturnErrorValuesAsNull",
             "",
             "annotation __PBI_TimeIntelligenceEnabled = 0",
             f'annotation PBI_QueryOrder = ["{expr_name}"]',
             ""]
    model += [f"ref table {q(tdisp[n])}" for n in table_names]
    model += [""]
    parts["definition/model.tmdl"] = "\n".join(model)

    # -- tables/*.tmdl
    for t in tables:
        name = t["name"]
        prof = PROFILES[name]
        hidden = set(t.get("hidden_columns", []))
        fmts = t.get("column_format_strings", {})
        tname = tdisp[name]
        lines = [f"table {q(tname)}", f"\tlineageTag: {lt()}"]
        if t.get("description"):
            lines.insert(1, f"\t/// {t['description']}")
        if dd.get("table") == name and dd.get("mark_as_date_table", True) and dd.get("date_column"):
            lines.append("\tdataCategory: Time")
        lines.append("")
        for m in measures_by_table.get(name, []):
            dax = m["dax_deployed"]
            if m.get("description"):
                lines.append(f"\t/// {m['description']}")
            if "\n" in dax:
                lines.append(f"\tmeasure {q(m['name'])} =\n\t\t\t" + "\n\t\t\t".join(dax.splitlines()))
            else:
                lines.append(f"\tmeasure {q(m['name'])} = {dax}")
            if m.get("format_string"):
                lines.append(f"\t\tformatString: {m['format_string']}")
            lines.append(f"\t\tlineageTag: {lt()}")
            lines.append("")
        for c in prof["columns"]:
            if not c["supported"]:
                continue
            cname = c["name"]
            is_key = cname in hidden or c.get("is_candidate_key") or cname == t.get("primary_key")
            lines.append(f"\tcolumn {q(cdisp[name][cname])}")
            lines.append(f"\t\tdataType: {c['tmdl_type']}")
            if cname in hidden:
                lines.append("\t\tisHidden")
            if dd.get("table") == name and cname == dd.get("date_column"):
                lines.append("\t\tisKey")
            if cname in fmts:
                lines.append(f"\t\tformatString: {fmts[cname]}")
            lines.append(f"\t\tlineageTag: {lt()}")
            lines.append(f"\t\tsummarizeBy: {'none' if (is_key or not c['is_numeric']) else 'sum'}")
            lines.append(f"\t\tsourceColumn: {cname}")
            lines.append("")
            lines.append("\t\tannotation SummarizationSetBy = Automatic")
            lines.append("")
        lines.append(f"\tpartition {q(tname)} = entity")
        lines.append("\t\tmode: directLake")
        lines.append("\t\tsource")
        lines.append(f"\t\t\tentityName: {name}")
        if SCHEMA_NAME or DIRECT_LAKE_MODE == "sql_endpoint":
            lines.append(f"\t\t\tschemaName: {SCHEMA_NAME or 'dbo'}")
        lines.append(f"\t\t\texpressionSource: {expr_name}")
        lines.append("")
        lines.append("\tannotation PBI_ResultType = Table")
        lines.append("")
        parts[f"definition/tables/{tname}.tmdl"] = "\n".join(lines)

    # -- relationships.tmdl
    rel_lines = []
    for r in rec["relationships"]:
        rel_lines.append(f"relationship {uuid.uuid4()}")
        if r.get("cardinality") == "oneToOne":
            rel_lines.append("\tfromCardinality: one")
            rel_lines.append("\ttoCardinality: one")
        if r.get("cross_filter") == "both":
            rel_lines.append("\tcrossFilteringBehavior: bothDirections")
        if r.get("is_active", True) is False:
            rel_lines.append("\tisActive: false")
        rel_lines.append(f"\tfromColumn: {q(tdisp[r['from_table']])}.{q(cdisp[r['from_table']][r['from_column']])}")
        rel_lines.append(f"\ttoColumn: {q(tdisp[r['to_table']])}.{q(cdisp[r['to_table']][r['to_column']])}")
        rel_lines.append("")
    parts["definition/relationships.tmdl"] = "\n".join(rel_lines)

    parts["definition.pbism"] = json.dumps({"version": "4.0", "settings": {}}, indent=2)
    return parts

TMDL_PARTS = build_tmdl(REC)
for p in sorted(TMDL_PARTS):
    print(f"── {p} ({len(TMDL_PARTS[p]):,} chars)")
print()
print(TMDL_PARTS["definition/relationships.tmdl"][:1500])
first_table = next(p for p in sorted(TMDL_PARTS) if p.startswith("definition/tables/"))
print(TMDL_PARTS[first_table][:3000])

> **Display names.** In TMDL the table/column *name* is what report authors see, while `entityName`
> and `sourceColumn` point at the lakehouse objects. The builder above renames objects to the LLM's
> display names and rewrites every `Table[Column]` reference in the measures to match.

## 12. Write recommendation + TMDL to the lakehouse Files area

In [ ]:
OUT_ROOT = (f"abfss://{WORKSPACE_ID}@onelake.dfs.fabric.microsoft.com/{LAKEHOUSE_ID}"
            f"/Files/semantic_model_recommendations/{re.sub(r'[^A-Za-z0-9_-]+', '_', MODEL_NAME)}")
if WRITE_TMDL:
    notebookutils.fs.put(f"{OUT_ROOT}/recommendation.json", json.dumps(REC, indent=2, default=str), True)
    notebookutils.fs.put(f"{OUT_ROOT}/recommendation.md", SUMMARY_MD, True)
    notebookutils.fs.put(f"{OUT_ROOT}/profile.json", json.dumps(PROFILES, indent=2, default=str), True)
    for path, content in TMDL_PARTS.items():
        notebookutils.fs.put(f"{OUT_ROOT}/{MODEL_NAME}.SemanticModel/{path}", content, True)
    print(f"Written to {OUT_ROOT}")
    print("Download the *.SemanticModel folder to open it as a PBIP in Power BI Desktop, "
          "or set DEPLOY_TO_WORKSPACE = True to publish directly.")

## 13. (Optional) Deploy the semantic model to this workspace

Uses the Fabric REST API *Create Semantic Model with definition*. Requires Contributor or higher
on the workspace. If a model with the same name already exists, its definition is updated instead.

In [ ]:
def fabric_api(method, url, token, **kw):
    import requests
    r = requests.request(method, url, headers={"Authorization": f"Bearer {token}"}, **kw)
    if r.status_code == 202:  # long-running operation
        op = r.headers["x-ms-operation-id"]; retry = int(r.headers.get("Retry-After", "5"))
        while True:
            time.sleep(retry)
            s = requests.get(f"https://api.fabric.microsoft.com/v1/operations/{op}",
                             headers={"Authorization": f"Bearer {token}"}).json()
            if s["status"] in ("Succeeded", "Failed"):
                if s["status"] == "Failed":
                    raise RuntimeError(f"Operation failed: {s}")
                return s
    r.raise_for_status()
    return r.json() if r.text else {}

def deploy(parts):
    token = notebookutils.credentials.getToken("pbi")
    base = f"https://api.fabric.microsoft.com/v1/workspaces/{WORKSPACE_ID}/semanticModels"
    definition = {"parts": [{"path": p, "payloadType": "InlineBase64",
                             "payload": base64.b64encode(c.encode("utf-8")).decode()} for p, c in parts.items()]}
    existing = [m for m in fabric_api("GET", base, token).get("value", []) if m["displayName"] == MODEL_NAME]
    if existing:
        mid = existing[0]["id"]
        print(f"Updating existing semantic model {MODEL_NAME} ({mid})")
        fabric_api("POST", f"{base}/{mid}/updateDefinition", token, json={"definition": definition})
    else:
        print(f"Creating semantic model {MODEL_NAME}")
        res = fabric_api("POST", base, token, json={"displayName": MODEL_NAME,
                                                     "description": REC.get("model_summary", "")[:250],
                                                     "definition": definition})
        mid = res.get("id") or next(m["id"] for m in fabric_api("GET", base, token)["value"]
                                    if m["displayName"] == MODEL_NAME)
    print(f"Done → https://app.fabric.microsoft.com/groups/{WORKSPACE_ID}/datasets/{mid}/details")
    return mid

if DEPLOY_TO_WORKSPACE:
    MODEL_ID = deploy(TMDL_PARTS)
else:
    print("DEPLOY_TO_WORKSPACE is False — skipping. Review the recommendation above first.")

## Next steps after deployment

- Open the model in the workspace and run a quick check, for example `EVALUATE { [Total Sales] }`
  in a DAX query view, to confirm Direct Lake column mappings resolve.
- If a relationship was flagged with a **type mismatch**, fix the column type in the lakehouse
  (cast in the Silver/Gold notebook) and rerun; Direct Lake cannot cast.
- Add RLS roles, hierarchies, and calculation groups in Power BI Desktop / web modeling as needed.
- Rerun this notebook after schema changes; the deploy cell updates the existing model in place.